# BaIP: hold-out experiment (exp/holdout-aug-rare), steps 3 to 5

Three runs on the fixed 70/15/15 split (`splits/holdout_70_15_15_v1.json`, 726 samples, 10 classes), one after another:

1. `crop448_ho_base`: baseline, no augmentation, no oversampling
2. `crop448_ho_aug`: plus augmentation (flips, 90° rotations, brightness, contrast, colour)
3. `crop448_ho_aug_os`: plus rare-fibre oversampling (repeat factor sampling, cap 5)

**Before you start:** Runtime > Change runtime type > **L4 GPU**. Keep this tab open.

**Run the cells top to bottom, once.** Cell 6 starts the queue in the background and returns at once. Use cell 7 to check progress.

**Estimate:** about 20 to 30 minutes per run, 1 to 1.5 hours in total, about 3 to 4 compute units.

**If Colab disconnects:** reconnect to an L4, run cells 1 to 4, then cell 6 again. Finished runs are skipped.

In [ ]:
# 1. Settings
BRANCH = 'exp/holdout-aug-rare'
QUEUE = 'configs/holdout/queue.txt'
PATHS = 'configs/paths_colab.yaml'
REPO = '/content/repo'
RESULTS = '/content/drive/MyDrive/BaIP/results'

In [ ]:
# 2. GPU check (must say NVIDIA L4) and Google Drive
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Code: clone (first time) or update, check out the branch, install the two extra packages
import os
if not os.path.isdir(REPO):
    !git clone -q https://github.com/merijnmolenaar/Textile_Classification_CNN_Merijn.git {REPO}
!cd {REPO} && git fetch -q && git checkout -q {BRANCH} && git pull -q && git log -1 --format='commit %h  %s'
!pip -q install pyyaml openpyxl
os.chdir(REPO)

In [ ]:
# 4. Data check: expect 2208 images in total (any split over Images / Images-2), labels True, split file True
import glob, yaml
p = yaml.safe_load(open(PATHS))['data']
n = sum(len(glob.glob(os.path.join(d, '*.jpg'))) for d in p['image_dirs'] if os.path.isdir(d))
print('images:', n, '(expect 2208)')
print('labels:', os.path.exists(p['labels_xlsx']))
print('split file:', os.path.exists('splits/holdout_70_15_15_v1.json'))

In [ ]:
# 5. Smoke test (about 2 minutes): the oversampling run (includes augmentation), 1 epoch of 3 batches,
#    into results/smoke. Check: '726 matched to labels', 10 classes, samples 504 / 111 / 111,
#    and an 'oversampling (expected sample draws per epoch)' line. 0% accuracy is expected.
!python scripts/train.py configs/holdout/crop448_ho_aug_os.yaml --paths {PATHS} \
    --set train.epochs=1 --set train.max_batches=3 --set run_dir={RESULTS}/smoke/holdout_aug_os
!python scripts/run_queue.py {QUEUE} --paths {PATHS} --dry-run

In [ ]:
# 6. Start the queue in the background. The cell returns at once; the queue keeps running.
#    Running it again while the queue is still busy does nothing (it checks first).
import subprocess, sys
busy = subprocess.run(['pgrep', '-f', 'scripts/run_queue.py'], capture_output=True, text=True).stdout.split()
if busy:
    print('queue already running, pid', busy)
else:
    os.makedirs(f'{RESULTS}/queue_logs', exist_ok=True)
    out = open(f'{RESULTS}/queue_logs/queue_console.log', 'a')
    proc = subprocess.Popen([sys.executable, '-u', 'scripts/run_queue.py', QUEUE, '--paths', PATHS],
                            stdout=out, stderr=subprocess.STDOUT, cwd=REPO, start_new_session=True)
    print('queue started, pid', proc.pid, '| progress: run cell 7')

In [ ]:
# 7. Progress (run whenever you like): finished runs, current run, latest epoch
import pathlib
st = pathlib.Path(f'{RESULTS}/queue_logs/queue_status.txt')
print(st.read_text()[-1200:] if st.exists() else 'no status yet')
hist = sorted(glob.glob(f'{RESULTS}/holdout/*/history.csv'), key=os.path.getmtime)
if hist:
    h = hist[-1]
    last = open(h).read().strip().splitlines()[-1].split(',')
    print(f'\nlatest: {h.split("/")[-2]} | epoch {last[0]} | val sample acc {float(last[6])*100:.1f}% | {float(last[8]):.0f} s/epoch')
print('queue running:', bool(subprocess.run(['pgrep', '-f', 'scripts/run_queue.py'], capture_output=True).stdout))
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

**When done** (cell 7 shows `queue finished: 3/3 OK`): Runtime > **Disconnect and delete runtime**, so it stops using compute units. Results: `MyDrive/BaIP/results/holdout/<run>/`, weights: `MyDrive/BaIP/checkpoints/holdout_<run>.pt`, logs: `MyDrive/BaIP/results/queue_logs/`.